In [ ]:
# === CELL 1 (NB5) — Reload/Resume ===
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json
DRIVE_ROOT = "/content/drive/MyDrive/FPX_Project"
sys.path.insert(0, os.path.join(DRIVE_ROOT, "utils"))
import checkpoint_utils as cu

config = cu.load_json("config", "project_config")
print("=== CELL 1 (NB5) === Resumed FPX project.")
print(f"Current phase: {config.get('current_phase')}")
cu.project_status()

Mounted at /content/drive
=== CELL 1 (NB5) === Resumed FPX project.
Current phase: P4_complete
=== FPX Project Drive Status ===
config: 2 file(s)
   - fault_config.json
   - project_config.json
data: 2 file(s)
   - client_partition.pkl
   - data_split.pkl
checkpoints/fl_sim: 21 file(s)
   - latest_round_state.pkl
   - round_001.pkl
   - round_002.pkl
   - round_003.pkl
   - round_004.pkl
   - round_005.pkl
   - round_006.pkl
   - round_007.pkl
   - round_008.pkl
   - round_009.pkl
checkpoints/signals: 2 file(s)
   - fault_config_used.json
   - signal_timeline.pkl
checkpoints/interface: 1 file(s)
   - generated_explanations.json
checkpoints/eval: 0 file(s)
outputs: 3 file(s)
   - fl_training_curve.png
   - p2_summary.json
   - signal_timeline.png
logs: 1 file(s)
   - project_log.txt


In [ ]:
# === CELL 2 (NB5) — Load signals + explanations ===
signal_history = cu.load_pickle("checkpoints/signals", "signal_timeline")
briefings = cu.load_json("checkpoints/interface", "generated_explanations")

if signal_history is None or briefings is None:
    raise RuntimeError("Missing P3/P4 outputs — run Notebooks 3 and 4 first.")

print(f"=== CELL 2 (NB5) === Loaded {len(signal_history)} rounds of signals and briefings.")

=== CELL 2 (NB5) === Loaded 15 rounds of signals and briefings.


In [ ]:
# === CELL 3 (NB5) — Scenario definitions + raw-numbers view builder ===
SCENARIOS = [
    {"id": "S1_quiet_warmup", "round": 3, "label": "Normal round, early in training"},
    {"id": "S2_early_signal", "round": 8, "label": "First unusual signal, not yet acted on"},
    {"id": "S3_fresh_exclusion", "round": 9, "label": "Just down-weighted (first round)"},
    {"id": "S4_sustained_exclusion", "round": 15, "label": "Long-standing down-weighted client"},
]

def build_raw_numbers_view(record):
    """The 'bare dashboard' baseline condition - same underlying data as the
    briefing, presented as raw structured numbers with no natural-language narration.
    This must be a FAIR baseline: same information, no framing advantage either way."""
    lines = [f"Round {record['round']} - Raw Signal Data", "-" * 40]
    lines.append(f"Global test accuracy: {record['test_accuracy']:.4f}")
    lines.append("")
    lines.append(f"{'Site':<6}{'CosSim':>10}{'Z-score':>10}{'EffShare':>10}{'Excluded':>10}")
    for i in range(len(record["z_score"])):
        lines.append(
            f"{i:<6}{record['cos_sim'][i]:>10.3f}{record['z_score'][i]:>10.2f}"
            f"{record['effective_share'][i]:>10.3f}{str(record['excluded'][i]):>10}"
        )
    return "\n".join(lines)

scenario_materials = {}
for s in SCENARIOS:
    idx = s["round"] - 1
    scenario_materials[s["id"]] = {
        "round": s["round"],
        "label": s["label"],
        "raw_view": build_raw_numbers_view(signal_history[idx]),
        "briefing_view": briefings[idx]["briefing_text"]
    }

cu.save_json(scenario_materials, "checkpoints/eval", "scenario_materials")
print("=== CELL 3 (NB5) === Built materials for", len(scenario_materials), "scenarios.")
print(scenario_materials["S3_fresh_exclusion"]["raw_view"])

[2026-09-03 06:48:15] Saved JSON checkpoint: checkpoints/eval/scenario_materials.json
=== CELL 3 (NB5) === Built materials for 4 scenarios.
Round 9 - Raw Signal Data
----------------------------------------
Global test accuracy: 0.9474

Site      CosSim   Z-score  EffShare  Excluded
0          0.032     -2.15     0.002      True
1          0.561      0.06     0.141     False
2          0.463     -0.35     0.107     False
3          0.470     -0.31     0.211     False
4          0.711      0.69     0.109     False
5          0.926      1.59     0.136     False
6          0.635      0.37     0.150     False
7          0.567      0.09     0.143     False


In [ ]:
# === CELL 4 (NB5) — Counterbalanced participant assignment ===
ids = [s["id"] for s in SCENARIOS]

# Exhaustive fixed 2-2 splits (3 partitions x 2 orientations = 6). Cycling through all 6
# guarantees exact balance - every scenario appears equally often in each condition,
# regardless of sample size or luck.
FIXED_SPLITS = [
    {"raw": [ids[0], ids[1]], "briefing": [ids[2], ids[3]]},
    {"raw": [ids[2], ids[3]], "briefing": [ids[0], ids[1]]},
    {"raw": [ids[0], ids[2]], "briefing": [ids[1], ids[3]]},
    {"raw": [ids[1], ids[3]], "briefing": [ids[0], ids[2]]},
    {"raw": [ids[0], ids[3]], "briefing": [ids[1], ids[2]]},
    {"raw": [ids[1], ids[2]], "briefing": [ids[0], ids[3]]},
]

def build_counterbalanced_assignment(num_participants):
    assignments = []
    for p in range(num_participants):
        split = FIXED_SPLITS[p % len(FIXED_SPLITS)]
        order = "raw_first" if p % 2 == 0 else "briefing_first"
        assignments.append({
            "participant_id": f"P{p+1:02d}",
            "raw_numbers_scenarios": split["raw"],
            "briefing_scenarios": split["briefing"],
            "presentation_order": order
        })
    return assignments

NUM_PARTICIPANTS = 8  # adjust to however many you actually recruit
assignments = build_counterbalanced_assignment(NUM_PARTICIPANTS)
cu.save_json(assignments, "checkpoints/eval", "participant_assignments")

for a in assignments:
    print(a)

[2026-09-03 06:48:29] Saved JSON checkpoint: checkpoints/eval/participant_assignments.json
{'participant_id': 'P01', 'raw_numbers_scenarios': ['S1_quiet_warmup', 'S2_early_signal'], 'briefing_scenarios': ['S3_fresh_exclusion', 'S4_sustained_exclusion'], 'presentation_order': 'raw_first'}
{'participant_id': 'P02', 'raw_numbers_scenarios': ['S3_fresh_exclusion', 'S4_sustained_exclusion'], 'briefing_scenarios': ['S1_quiet_warmup', 'S2_early_signal'], 'presentation_order': 'briefing_first'}
{'participant_id': 'P03', 'raw_numbers_scenarios': ['S1_quiet_warmup', 'S3_fresh_exclusion'], 'briefing_scenarios': ['S2_early_signal', 'S4_sustained_exclusion'], 'presentation_order': 'raw_first'}
{'participant_id': 'P04', 'raw_numbers_scenarios': ['S2_early_signal', 'S4_sustained_exclusion'], 'briefing_scenarios': ['S1_quiet_warmup', 'S3_fresh_exclusion'], 'presentation_order': 'briefing_first'}
{'participant_id': 'P05', 'raw_numbers_scenarios': ['S1_quiet_warmup', 'S4_sustained_exclusion'], 'briefing

In [ ]:
# === CELL 5 (NB5) — Per-participant session packets ===
def build_participant_packet(assignment, scenario_materials):
    packet = {"participant_id": assignment["participant_id"], "presentation_order": assignment["presentation_order"], "items": []}
    for sid in assignment["raw_numbers_scenarios"]:
        m = scenario_materials[sid]
        packet["items"].append({"scenario_id": sid, "condition": "raw_numbers", "round": m["round"], "content": m["raw_view"]})
    for sid in assignment["briefing_scenarios"]:
        m = scenario_materials[sid]
        packet["items"].append({"scenario_id": sid, "condition": "briefing", "round": m["round"], "content": m["briefing_view"]})
    if assignment["presentation_order"] == "briefing_first":
        packet["items"] = packet["items"][2:] + packet["items"][:2]
    return packet

all_packets = [build_participant_packet(a, scenario_materials) for a in assignments]
cu.save_json(all_packets, "checkpoints/eval", "participant_packets")

print("=== CELL 5 (NB5) === Example packet for P01: ===")
print(json.dumps(all_packets[0], indent=2))

[2026-09-03 06:48:42] Saved JSON checkpoint: checkpoints/eval/participant_packets.json
=== CELL 5 (NB5) === Example packet for P01: ===
{
  "participant_id": "P01",
  "presentation_order": "raw_first",
  "items": [
    {
      "scenario_id": "S1_quiet_warmup",
      "condition": "raw_numbers",
      "round": 3,
      "content": "Round 3 - Raw Signal Data\n----------------------------------------\nGlobal test accuracy: 0.7368\n\nSite      CosSim   Z-score  EffShare  Excluded\n0          0.340     -0.70     0.035     False\n1          0.544      0.26     0.136     False\n2          0.190     -1.41     0.103     False\n3          0.205     -1.34     0.204     False\n4          0.652      0.77     0.105     False\n5          0.840      1.66     0.132     False\n6          0.594      0.50     0.145     False\n7          0.544      0.26     0.138     False"
    },
    {
      "scenario_id": "S2_early_signal",
      "condition": "raw_numbers",
      "round": 8,
      "content": "Round 8 - Raw

In [9]:
# === CELL 6 (NB5) — Study protocol document [PATCHED: fairness glossary] ===
protocol = """
FPX FORMATIVE EVALUATION - MODERATOR SCRIPT
=============================================
Format: Live video call, ~15-20 min, recorded with consent
Population: ML/CS grad students or researchers (proxy population - stated as a limitation)

INTRO (2 min)
- Explain: "You're looking at a federated learning system where multiple hospitals
  train a shared model together. I'll show you snapshots from the training process.
  Imagine you're the person responsible for making sure this training run is healthy."
- Read the glossary below ALOUD to every participant, regardless of condition -
  this keeps the comparison fair (both conditions start from equal understanding
  of what the system measures; only the communication format differs):

    "Cosine similarity - how similar a site's update is to the group's average
     update this round. Closer to 1 means very similar; near 0 or negative means
     quite different.
     Z-score - how unusual a site's similarity is compared to its peers this
     round, in standard deviations. Below about -1 is starting to look unusual;
     below -2 is quite unusual.
     Effective share - how much a site's update actually counted toward this
     round's result, after any down-weighting.
     Excluded - whether the system significantly reduced this site's influence
     this round due to persistent unusual behavior."

- Remind: think aloud - say what you're noticing and thinking as you go.
- Get recording consent.

FOR EACH SCENARIO (4 total, ~3-4 min each):
  Show the material (raw numbers OR briefing, per their assigned condition).

  Q1 [Comprehension - open]: "In your own words, what's happening in this round?"
  Q2 [Comprehension - check]: "Is any site currently being down-weighted by the system? (Yes/No/Not sure)"
       -> record correct/incorrect against ground truth (see scenario_materials round data)
  Q3 [Trust]: "On a scale of 1-7, how much would you trust this round's model?" + "Why?"
  Q4 [Actionability]: "What, if anything, would you do next?"
  Q5 [Confidence]: "How confident are you in that decision, 1-7?"

  Moderator notes: approximate time-to-first-correct-understanding (stopwatch,
  starts when material is shown, stops when they articulate the core fact correctly)

WRAP-UP (3 min)
- "Overall, which format (numbers or plain-language) was easier to reason with, and why?"
- "Was there anything you didn't trust or found confusing?"
- Thank + explain proxy-population/limitation context if they ask why it's not a real hospital scenario.
"""

protocol_path = os.path.join(DRIVE_ROOT, "outputs", "study_protocol.txt")
with open(protocol_path, "w") as f:
    f.write(protocol)

print("=== CELL 6 (NB5) === Protocol saved (with fairness glossary) to outputs/study_protocol.txt")
print(protocol)

=== CELL 6 (NB5) === Protocol saved (with fairness glossary) to outputs/study_protocol.txt

FPX FORMATIVE EVALUATION - MODERATOR SCRIPT
Format: Live video call, ~15-20 min, recorded with consent
Population: ML/CS grad students or researchers (proxy population - stated as a limitation)

INTRO (2 min)
- Explain: "You're looking at a federated learning system where multiple hospitals
  train a shared model together. I'll show you snapshots from the training process.
  Imagine you're the person responsible for making sure this training run is healthy."
- Read the glossary below ALOUD to every participant, regardless of condition -
  this keeps the comparison fair (both conditions start from equal understanding
  of what the system measures; only the communication format differs):

    "Cosine similarity - how similar a site's update is to the group's average
     update this round. Closer to 1 means very similar; near 0 or negative means
     quite different.
     Z-score - how unusual a s

In [ ]:
# === CELL 7 (NB5) — Response data template ===
response_template = {
    "participant_id": "", "scenario_id": "", "condition": "", "round": 0,
    "q1_comprehension_text": "", "q2_comprehension_correct": None,  # True/False
    "q3_trust_score": None, "q3_trust_reason": "",
    "q4_action_text": "", "q5_confidence_score": None,
    "time_to_understanding_sec": None, "moderator_notes": ""
}

cu.save_json(response_template, "checkpoints/eval", "response_template")
cu.save_json([], "checkpoints/eval", "collected_responses")  # empty log, ready to append real data into

config["current_phase"] = "P5_materials_ready"
cu.save_json(config, "config", "project_config")

print("=== CELL 7 (NB5) === Response template + empty response log saved.")
print("Ready to run sessions. Log each response as a dict matching this template,")
print("append to checkpoints/eval/collected_responses.json, and re-save via cu.save_json.")

[2026-09-03 06:49:14] Saved JSON checkpoint: checkpoints/eval/response_template.json
[2026-09-03 06:49:14] Saved JSON checkpoint: checkpoints/eval/collected_responses.json
[2026-09-03 06:49:14] Saved JSON checkpoint: config/project_config.json
=== CELL 7 (NB5) === Response template + empty response log saved.
Ready to run sessions. Log each response as a dict matching this template,
append to checkpoints/eval/collected_responses.json, and re-save via cu.save_json.
